# After the Water: Pakistan Floods 2022
## Notebook A · Session 1 — Reading the flood data

**How it works.** Work in pairs and swap the typist at every task. Run the cells from the top (Shift+Enter). Where it says `TODO`, write your code, run the cell, then run the **test cell** under it and aim for all `[PASS]`. Type your answers in the *My answer* cells.

Keep the **Session 1 Helper Sheet** open. Where the data comes from: `DATA_SOURCES.md` in the Introduction on the course page.

| Task | Function | What it does |
| --- | --- | --- |
| 1 | `clean_name(name)` | remove a footnote like "(2)" |
| 2 | `classify_row(name)` | junk, country, province or district? |
| 3 | `count_kinds(rows)` | how many rows of each kind? |

## A story before we start

In summer 2022, monsoon rain in Pakistan was nearly three times the 30-year average. Large parts of Sindh and Balochistan became lakes.

Two counts of the damage:

- The government: about 33 million people affected.
- UN satellites: 23.7 million people in or near the water at the end of August, and still 8.3 million in mid-November.

Both numbers are real. They measure different things.

This week's question: after the 2022 floods, which districts were hit hardest, did help reach them, and when could people get back to their land? Today we start by reading the satellite data.

**Today's plan** (75 minutes)

```
 0-10 min   Story
10-20 min   Sections 1-2  Read the file; the 55 columns
20-30 min   Section 3     find_row together; Dadu week by week
30-55 min   Section 4     What kind of row is each row? Tasks 1-2
55-65 min   Section 5     Task 3 and the Islamabad mystery
65-75 min   Section 6     Save your work for Session 2; wrap-up
```

In [57]:
import csv
from helpers import banner, check, summary

banner("READY")
print("banner() is loaded. Let's look at the flood data.")


########################################################################
#  READY
########################################################################
banner() is loaded. Let's look at the flood data.


## Section 1 — Read the file with csv.reader

A CSV file is text: values separated by commas, one row per line. But some cells in this file have a comma or a line break inside them, so splitting at commas ourselves would cut them in two. csv.reader knows how to handle this, and gives back one list of strings per row.

In [58]:
banner("SECTION 1 — Reading with csv.reader")

FILE = r"C:\Users\ma12312\Documents\Case Studies\Case Study (2)\session1_student\session1_ma12312\floods2022_water_by_district.csv"

with open(FILE, "r", encoding="utf-8", newline="") as f:
    rows = list(csv.reader(f))

# print(rows)
print("Rows (including header):", len(rows))
print("Columns in the header:  ", len(rows[0]))
print("Columns in row 1:       ", len(rows[1]))
print("Row 1 starts with:", rows[1][:3])


########################################################################
#  SECTION 1 — Reading with csv.reader
########################################################################
Rows (including header): 175
Columns in the header:   55
Columns in row 1:        55
Row 1 starts with: ['Pakistan', '877964.6739099998', '231088952.39900398']


rows is a list of lists:

```
rows[0]      the header (the column names)
rows[1]      the first data row
rows[1][0]   the first cell of the first data row
```

Every value is a string, even the numbers (see the quotes around '877964.67...'). We turn them into numbers in Session 2.

## Section 2 — What are the 55 columns?

The first 8 column names, with their numbers. (Some names have a line break inside them; we replace it with a space.)

In [59]:
banner("SECTION 2a — The first 8 column names")

header = rows[0]
for i in range(len(header)):
    header[i] = header[i].replace("\n", " ")

for i in range(9):
    print(i, "|", header[i])
print("... and 47 more")


########################################################################
#  SECTION 2a — The first 8 column names
########################################################################
0 | Country/Province/District (1)
1 | Area of Country/ Province/ District/ (km2)
2 | Total population in  Province 
3 | Analysed area [01-31 Aug. 2022] (km2)
4 | Total Population in Analyzed area [01-31 Aug. 2022]
5 | Maximum flood water extent  [01-31 Aug. 2022] (km2)
6 | Population potentially exposed [01-31 Aug. 2022]
7 | Population potentially exposed in analyzed area  [01-31 Aug. 2022] (%)
8 | Analysed area (in cloud free zone) [25-31 Aug. 2022] (km2)
... and 47 more


**Reading the pattern** (together)

```
Column 0       place name
Column 1       area (km2)
Column 2       population
Columns 3-47   9 satellite windows, 5 columns each. We need 2 of the 5:
               flood water (km2) and people exposed.
Columns 48-54  change between windows (we don't use these)
```

Our 2 columns repeat every 5 columns, so we write their positions down once.

In [60]:
banner("SECTION 2b — Checking 3 of the 9 windows")

WINDOW_LABELS = ["01-31 Aug", "25-31 Aug", "01-07 Sep", "08-14 Sep", "15-21 Sep",
                 "26 Sep-02 Oct", "03-09 Oct", "11-17 Oct", "14-20 Nov"]
FLOOD_COLS   = [5, 10, 15, 20, 25, 30, 35, 40, 45]    # flood water extent (km2)
EXPOSED_COLS = [6, 11, 16, 21, 26, 31, 36, 41, 46]    # people exposed

# Check: does each position really hold the column we think it does?
for k in [0, 1, 8]:          # the first, second and last window
    flood_col = FLOOD_COLS[k]
    exposed_col = EXPOSED_COLS[k]

    print("Window:", WINDOW_LABELS[k])
    print("   column", flood_col, "=", header[flood_col])
    print("   column", exposed_col, "=", header[exposed_col])


########################################################################
#  SECTION 2b — Checking 3 of the 9 windows
########################################################################
Window: 01-31 Aug
   column 5 = Maximum flood water extent  [01-31 Aug. 2022] (km2)
   column 6 = Population potentially exposed [01-31 Aug. 2022]
Window: 25-31 Aug
   column 10 = Maximum flood water extent  [25-31 Aug. 2022] (km2)
   column 11 = Population potentially exposed in analyzed area [25-31 Aug. 2022]
Window: 14-20 Nov
   column 45 = Maximum flood water extent  [14 - 20 Nov. 2022] (km2)
   column 46 = Population potentially exposed [14 - 20 Nov. 2022]


**Question 1:** The first window, "01-31 Aug", covers a whole month; the others cover about a week. Why could that be a problem when we compare windows?


**Answer**: We are comparing a big window with a small one which could lead to inconsistent diffrences

## Section 3 — Together: find one district's row

We write this one together: the instructor types, you help. It looks through rows and gives back the row whose first cell matches a name.

In [61]:
banner("SECTION 3a — TOGETHER: find_row")

def find_row(rows, name):
    """Return the first row whose first cell equals name, or None if none does."""
    # >>> TOGETHER: we write this as a class <<<
    for row in rows:
        if row[0] == name:
            return row
    return None


dadu = find_row(rows, "Dadu")
if dadu is None:
    print("find_row gave None for Dadu: it isn't working yet.")
else:
    print("find_row(rows, 'Dadu') starts with:", dadu[:3])
print("find_row(rows, 'Atlantis') gives:", find_row(rows, "Atlantis"))


########################################################################
#  SECTION 3a — TOGETHER: find_row
########################################################################
find_row(rows, 'Dadu') starts with: ['Dadu', '8015.597', '1846783.7820489998']
find_row(rows, 'Atlantis') gives: None


Now let's see how the water in Dadu (Sindh) changed over the weeks.

In [62]:
banner("SECTION 3b — Dadu, week by week")

if dadu is None:
    print("find_row isn't working yet: finish it above, then run this again.")
else:
    print("Dadu — land under water and people exposed")
    for k in range(len(WINDOW_LABELS)):
        km2 = dadu[FLOOD_COLS[k]]
        people = dadu[EXPOSED_COLS[k]]
        print(WINDOW_LABELS[k], ":", km2, "km2 under water,", people, "people exposed")


########################################################################
#  SECTION 3b — Dadu, week by week
########################################################################
Dadu — land under water and people exposed
01-31 Aug : 2627.515 km2 under water, 1262147.07166 people exposed
25-31 Aug : 2597.0730000000003 km2 under water, 1239852.658523 people exposed
01-07 Sep : 2503.227 km2 under water, 1213532.2721230001 people exposed
08-14 Sep : 2408.9040000000005 km2 under water, 1127861.4092020001 people exposed
15-21 Sep : 2303.034 km2 under water, 1060532.025413 people exposed
26 Sep-02 Oct : 2005.3819999999998 km2 under water, 917451.246878 people exposed
03-09 Oct : 1809.7700000000002 km2 under water, 819198.37995 people exposed
11-17 Oct : 1700.107 km2 under water, 761368.7506230001 people exposed
14-20 Nov : 1402.125 km2 under water, 596621.718724 people exposed


**Question 2:** If this trend continued, roughly when would Dadu's water be gone? Keep your guess: in the lab you'll make this forecast properly.

**My guess:** Mid of March

## Section 4 — What kind of row is each row?

The first 20 names in the file, with their positions in rows.

In [63]:
banner("SECTION 4 — The first 20 names in the file")

for i in range(1, 21):
    print(i, rows[i][0])


########################################################################
#  SECTION 4 — The first 20 names in the file
########################################################################
1 Pakistan
2 Azad Kashmir (2)
3 Bagh
4 Bhimber
5 Haveli
6 Jhelum Valley
7 Kotli
8 Mirpur
9 Muzaffarabad
10 Neelum
11 Poonch
12 Sudhnoti
13 Balochistan
14 Awaran
15 Barkhan
16 Chagai
17 Chaman
18 Dera Bugti
19 Duki
20 Gwadar


**Question 3:** Nothing on Bagh's row says which province it's in. How does a person know Bagh is in Azad Kashmir? And what would happen if someone sorted this file alphabetically?

**My answer:** When a value of col equals to a province until another value is spotted as a province all the other values under it are districts , all the districts, provinces will mix up 

## Task 1 — clean_name(name)

Some names end with a footnote marker, like "Azad Kashmir (2)". The (2) isn't part of the name.

**What to do**

Write clean_name(name) so that it:

1. removes spaces at the start and end, and
2. if the name ends with ")", cuts off everything from the last "(" onwards (then removes spaces again).

```
clean_name("Azad Kashmir (2)")   gives   "Azad Kashmir"
clean_name("  Dadu ")            gives   "Dadu"
```

**Useful tools**

```
.strip()   .endswith(")")   .rfind("(")
.rfind("(") gives the position of the last "(" in a string.
name[:k] keeps everything before position k.
```

In [64]:
def clean_name(name):
    """Remove surrounding spaces and a trailing footnote marker like "(2)"."""
    # >>> TODO (Task 1): write your code here <<<
    user = name.strip(' (2)')
    return user


In [65]:
# TEST — Task 1: clean_name      (given code: don't change this cell)
# Run the cell above first, and again every time you change clean_name.
results = []
results.append(check('clean_name("Azad Kashmir (2)")', clean_name("Azad Kashmir (2)"), "Azad Kashmir"))
results.append(check('clean_name("Gilgit Baltistan (2)")', clean_name("Gilgit Baltistan (2)"), "Gilgit Baltistan"))
results.append(check('clean_name("  Dadu ")', clean_name("  Dadu "), "Dadu"))
results.append(check('clean_name("Dadu")', clean_name("Dadu"), "Dadu"))
summary("Task 1", results)

  [PASS] clean_name("Azad Kashmir (2)") gave 'Azad Kashmir'
  [PASS] clean_name("Gilgit Baltistan (2)") gave 'Gilgit Baltistan'
  [PASS] clean_name("  Dadu ") gave 'Dadu'
  [PASS] clean_name("Dadu") gave 'Dadu'

  Task 1: 4 of 4 tests passed.
  Well done! Move on to the next task.


## Task 2 — classify_row(name)

classify_row takes a row's name (its first cell) and decides which of four kinds the row is:

```
"junk"      e.g. "Population data: WorldPop [2020]", "(1) The designa..."
            -> it's empty, OR contains ":", OR starts with "("
"country"   e.g. "Pakistan"
            -> its cleaned name is "Pakistan"
"province"  e.g. "Sindh", "Azad Kashmir (2)"
            -> its cleaned name is in PROVINCES
"district"  e.g. "Dadu", "Bagh"
            -> anything else
```

**What to do**

Write classify_row(name) to return one of these four strings. Use your clean_name from Task 1. Check for junk first.

In [66]:
PROVINCES = {"Azad Kashmir", "Balochistan", "Gilgit Baltistan", "Islamabad",
             "Khyber Pakhtunkhwa", "Punjab", "Sindh"}


def classify_row(name):
    """Return "junk", "country", "province" or "district" for a first-column value."""
    # >>> TODO (Task 2): write your code here <<<
    junk = ["",':',"("]
    country = "Pakistan"
    if name == country:
        return "country"
    if clean_name(name) in PROVINCES:
        return "province"
    for i in name:
            if i in junk:
                return "junk"
    if name == '':
         return "junk"
    else:
        return "district"

In [67]:
# TEST — Task 2: classify_row      (given code: don't change this cell)
# Run the cell above first, and again every time you change classify_row.
results = []
results.append(check('classify_row("Pakistan")', classify_row("Pakistan"), "country"))
results.append(check('classify_row("Azad Kashmir (2)")', classify_row("Azad Kashmir (2)"), "province"))
results.append(check('classify_row("Sindh")', classify_row("Sindh"), "province"))
results.append(check('classify_row("Dadu")', classify_row("Dadu"), "district"))
results.append(check('classify_row("Population data: WorldPop [2020]")',
                     classify_row("Population data: WorldPop [2020]"), "junk"))
results.append(check('classify_row("(1) The designations employed and...")',
                     classify_row("(1) The designations employed and..."), "junk"))
results.append(check('classify_row("")', classify_row(""), "junk"))
summary("Task 2", results)

  [PASS] classify_row("Pakistan") gave 'country'
  [PASS] classify_row("Azad Kashmir (2)") gave 'province'
  [PASS] classify_row("Sindh") gave 'province'
  [PASS] classify_row("Dadu") gave 'district'
  [PASS] classify_row("Population data: WorldPop [2020]") gave 'junk'
  [PASS] classify_row("(1) The designations employed and...") gave 'junk'
  [PASS] classify_row("") gave 'junk'

  Task 2: 7 of 7 tests passed.
  Well done! Move on to the next task.


## Section 5 — Count what's in the file

### Task 3 — count_kinds(rows)

**What to do**

Write count_kinds(rows). Run classify_row on the first cell of every data row (skip the header, rows[0]) and count the results in a dictionary, e.g. {"country": 1, "district": ..., ...}

In [68]:
def count_kinds(rows):
    """Return a dictionary: kind of row -> how many rows of that kind."""
    # >>> TODO (Task 3): write your code here <<<
    dic = {}
    for row in rows[1:]:
        x = classify_row(row[0])
        if x not in dic:
            dic[x] = 1
        else:
            dic[x] += 1
    return dic

In [69]:
# TEST — Task 3: count_kinds      (given code: don't change this cell)
# Run the cell above first, and again every time you change count_kinds.
# count_kinds uses classify_row, so Task 2 needs to pass first.
results = []
results.append(check("count_kinds(rows)", count_kinds(rows),
                     {"country": 1, "province": 8, "district": 159, "junk": 6}))
summary("Task 3", results)

  [PASS] count_kinds(rows) gave {'country': 1, 'province': 8, 'district': 159, 'junk': 6}

  Task 3: 1 of 1 tests passed.
  Well done! Move on to the next task.


In [70]:
banner("TASK 3 — Counting each kind of row")

counts = count_kinds(rows)
print("count_kinds(rows) gives:", counts)


########################################################################
#  TASK 3 — Counting each kind of row
########################################################################
count_kinds(rows) gives: {'country': 1, 'province': 8, 'district': 159, 'junk': 6}


Pakistan has 7 provinces and regions. How many province rows did you count?

**Answer** : 8

## Given code — the Islamabad mystery

Where are the province rows? This prints every row your classify_row calls a "province".

In [71]:
banner("SECTION 5 — Every province row")

if classify_row("Sindh") is None:
    print("classify_row isn't written yet: finish Task 2, then run this again.")
else:
    for i in range(1, len(rows)):
        if classify_row(rows[i][0]) == "province":
            print(i, rows[i][:3])


########################################################################
#  SECTION 5 — Every province row
########################################################################
2 ['Azad Kashmir (2)', '11471.7228', '3714094.716915']
13 ['Balochistan', '347667.29299999995', '8411096.857369']
49 ['Gilgit Baltistan (2)', '70226.29699999999', '1118618.5811029999']
64 ['Islamabad', '904.062', '2041786.35146']
65 ['Islamabad', '904.062', '2041786.35146']
66 ['Khyber Pakhtunkhwa', '100987.62250000001', '37461970.332799']
102 ['Punjab', '205759.37899999993', '127820391.202601']
139 ['Sindh', '140948.29760999998', '50520994.35675699']


**Question 4:** Islamabad appears twice in a row (rows 64 and 65), with exactly the same numbers. Is it a mistake, a duplicate, or something else?

**Answer** : Mistake

Keep this in mind: in Session 2 your code will have to handle Islamabad.

## Section 6 — Save your work for Session 2

Given code: just run the next two cells. They use your clean_name and classify_row to save a cleaner copy of the data for Session 2.

**Rows:** 174 -> 168

- kept: the country, the 8 province rows and the 159 district rows
- dropped: the 6 junk rows at the end (4 lines about where the data came from, and 2 UN footnotes)

**Columns:** 55 -> 22

- added: "level" (first column): your classify_row's answer
- kept: the name (cleaned by your clean_name), area and population
- kept: flood water (km2) and people exposed, for each of the 9 windows
- dropped: the other 3 columns of each window (analysed area, population analysed, % exposed) and the 7 "change" columns at the end

The new columns, in order:

```
level | name | area_km2 | population | flood_km2 x 9 | exposed x 9
```

In [72]:
OUT = "s1_classified_rows.csv"


def make_new_header():
    """GIVEN: the 22 column names of the file we save."""
    new_header = ["level", "name", "area_km2", "population"]
    for label in WINDOW_LABELS:
        new_header.append("flood_km2 " + label)
    for label in WINDOW_LABELS:
        new_header.append("exposed " + label)
    return new_header


def make_new_row(row):
    """GIVEN: the cleaned version of one row (22 values), or None if it's junk."""
    kind = classify_row(row[0])
    if kind == "junk":
        return None
    new_row = [kind, clean_name(row[0]), row[1], row[2]]
    for c in FLOOD_COLS:
        new_row.append(row[c])
    for c in EXPOSED_COLS:
        new_row.append(row[c])
    return new_row

In [73]:
banner("SAVING s1_classified_rows.csv")

if classify_row("Sindh") is None or clean_name("Sindh") is None:
    print("Tasks 1 and 2 aren't written yet, so nothing was saved.")
else:
    if dadu is not None:
        print("One row, before and after:")
        print("   BEFORE (55 values):", dadu[:4], "...")
        print("   AFTER  (22 values):", make_new_row(dadu)[:4], "...")
        print()

    saved = 0
    with open(OUT, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(make_new_header())
        for row in rows[1:]:
            new_row = make_new_row(row)
            if new_row is not None:
                writer.writerow(new_row)
                saved = saved + 1

    print("Saved", saved, "rows to", OUT, "  (expected 168)")
    print()
    print("The first 3 lines of your new file:")
    with open(OUT, "r", encoding="utf-8") as f:
        for i in range(3):
            print("  ", f.readline().strip()[:100], "...")


########################################################################
#  SAVING s1_classified_rows.csv
########################################################################
One row, before and after:
   BEFORE (55 values): ['Dadu', '8015.597', '1846783.7820489998', '8015.597'] ...
   AFTER  (22 values): ['district', 'Dadu', '8015.597', '1846783.7820489998'] ...

Saved 168 rows to s1_classified_rows.csv   (expected 168)

The first 3 lines of your new file:
   level,name,area_km2,population,flood_km2 01-31 Aug,flood_km2 25-31 Aug,flood_km2 01-07 Sep,flood_km2 ...
   country,Pakistan,877964.6739099998,231088952.39900398,83557.111748835,68963.77902800002,59789.418790 ...
   province,Azad Kashmir,11471.7228,3714094.716915,99.45110404,57.460030999999994,57.874008,55.76745618 ...


Something wrong? Check the tests for Tasks 1-2. Or use the ready-made s1_classified_rows.csv in the Session 2 files.

## Wrap-up (last 3 minutes) — write one line for each

1. What surprised you today?
2. One thing you understand now that you didn't 75 minutes ago:
3. One thing that still confuses you (bring it to Session 2):

Next time: we turn these flat rows into folders inside folders (Pakistan -> provinces -> districts) and ask which districts were hit hardest.

In [74]:
banner("END OF SESSION 1")


########################################################################
#  END OF SESSION 1
########################################################################
